# Data Import and QC

Purpose: load a CITE-seq object, validate RNA/protein modalities, write QC tables, and save the standard benchmark AnnData layout.

Inputs: a 10x HDF5/MEX path, `.h5ad`, `.h5mu`, or `scvi:` dataset spec.

Outputs: `data/processed/pbmc5k_10x_citeseq_imported.h5ad`, QC tables, and QC figures.

Matching script: `scripts/import_data.py`.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "rarecell").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from rarecell.config import DATA_DIR, FIGURES_DIR, TABLES_DIR
from rarecell.io import (
    get_cell_labels,
    load_citeseq,
    make_candidate_target_population_table,
    make_dataset_summary,
    save_citeseq_object,
    save_summary_tables,
    summarize_cell_labels,
    to_internal_anndata,
)
from rarecell.plotting import plot_bar_counts, plot_qc_summary


In [ ]:
INPUT = "scvi:5k_pbmc_protein_v3_nextgem"
DATASET = "pbmc5k_10x_citeseq"
OUTPUT = DATA_DIR / "processed" / "pbmc5k_10x_citeseq_imported.h5ad"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
obj = load_citeseq(INPUT)
labels = get_cell_labels(obj)
summary = make_dataset_summary(obj)
summary["cell_type_counts"] = summarize_cell_labels(labels)
save_summary_tables(summary, TABLES_DIR)
plot_qc_summary(summary, FIGURES_DIR / "qc_summary.png")

if labels is not None:
    make_candidate_target_population_table(labels).to_csv(TABLES_DIR / "candidate_target_populations.csv", index=False)
    plot_bar_counts(labels, FIGURES_DIR / "cell_label_counts.png", "Cell counts by label", "Label", "Cells")

adata = to_internal_anndata(obj, dataset=DATASET)
save_citeseq_object(adata, OUTPUT)
OUTPUT
